In [2]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION
)



In [3]:
functions = [
    {
        "name": "get_pizza_info",
        "description": "Get name and price of a pizza of the restaurant",
        "parameters": {
            "type": "object",
            "properties": {
                "pizza_name": {
                    "type": "string",
                    "description": "The name of the pizza, e.g. Hawaii",
                },
            },
            "required": ["pizza_name"],
        },
    },
    {
        "name": "place_order",
        "description": "Place an order for a pizza of the restaurant",
        "parameters": {
            "type": "object",
            "properties": {
                "quantity": {
                    "type": "integer",
                    "description": "The number of pizzas you want to order",
                    "minimum": 1
                },
                "pizza_name": {
                    "type": "string",
                    "description": "The name of the pizza you want to order, e.g. Margherita",
                },
                "address": {
                    "type": "string",
                    "description": "The address where the pizza should be delivered",
                },
            },
            "required": ["pizza_name", "quantity", "address"],
        },
    }
]

In [4]:
template = """
You are an AI Chatbot having a conversation with a human.

Human: {human_input}
AI:
"""

# promt = PromptTemplate(input_variables=["human_input"], template=template)

# result = llm_openai.invoke(promt.format(human_input="How much does Pizza hawaii cost?"), functions=functions)

# messages = ChatPromptTemplate.from_messages([
#     SystemMessage(content="You are an AI Chatbot having a conversation with a human."),
#     HumanMessage(content="How much does Pizza hawaii cost?")
# ]
# )

messages = [
    SystemMessage(content="You are an AI Chatbot having a conversation with a human."),
    HumanMessage(content="I want to order two pizza hawaii to 123 Fakestreet")
]

result = llm_openai.invoke(messages, functions=functions)

# prompt = ChatPromptTemplate.from_messages([
#     ("system","You are an AI Chatbot having a conversation with a human."),
#     ("human", "How much does Pizza hawaii cost?")
# ])

# result = llm_openai.invoke(prompt, functions=functions)

print(result)


[04/09/26 11:47:54] INFO     HTTP Request: POST                                                     ]8;id=350606;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=379668;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='',
    additional_kwargs={
        'function_call': {
            'arguments': '{"quantity":2,"pizza_name":"Hawaii","address":"123 Fakestreet"}',
            'name': 'place_order'
        },
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 28,
            'prompt_tokens': 167,
            'total_tokens': 195,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_7a7fd0eb44',
        'id': 'chatcmpl-DSaewxgFvDzn80s4QDOquKr5lQDS2',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'function_call',
        'logprobs': None,
        'content_filter_results': {}
    },
    id='lc_run--019d705a-82dc-7372-b757-84f26c94d744-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 167,
        'output_tokens': 28,
        'total_tokens': 195,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [24]:
import json

def place_order(pizza_name: str, quantity: int, address: str) -> dict:
    # Simulate placing an order and return a confirmation
    return {
        "status": "success",
        "message": f"Order placed: {quantity} {pizza_name} pizza(s) to {address}"
    }

if hasattr(result, "additional_kwargs") and "function_call" in result.additional_kwargs:
    func_call = result.additional_kwargs["function_call"]
    func_name = func_call["name"]
    func_args = json.loads(func_call["arguments"])

    print(type(func_args))
    print(func_args)


    if func_name == "place_order":
        func_result = place_order(**func_args)

        print("Function called:", func_name)
        print("Arguments:", json.dumps(func_args, indent=4))
        print("Function result:", json.dumps(func_result, indent=4))
else:
    print("AI response:", result.content)

<class 'dict'>

{'quantity': 2, 'pizza_name': 'Hawaii', 'address': '123 Fakestreet'}

Function called: place_order

Arguments: {
    "quantity": 2,
    "pizza_name": "Hawaii",
    "address": "123 Fakestreet"
}

Function result: {
    "status": "success",
    "message": "Order placed: 2 Hawaii pizza(s) to 123 Fakestreet"
}

# How does it work with the vanilla API?

In [6]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

openai.api_type = "azure"
openai.azure_endpoint = AZURE_OPENAI_API_ENDPOINT
openai.api_version = AZURE_OPENAI_API_VERSION
openai.apikey = AZURE_OPENAI_API_KEY

response = openai.chat.completions.create(
    model = AZURE_OPENAI_API_DEPLOYMENT_4O,
    messages = [
        {"role": "system", "content": "You are an AI Chatbot having a conversation with a human."},
        {"role": "user", "content": "I want to order two pizza hawaii to 123 Fakestreet"}
    ],
    functions=functions
)

print(response)






[04/09/26 11:48:15] INFO     HTTP Request: POST                                                     ]8;id=777435;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=436174;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

ChatCompletion(
    id='chatcmpl-DSafGgcgkCie3Q4znnBCh8DHdXblj',
    choices=[
        Choice(
            finish_reason='function_call',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                refusal=None,
                role='assistant',
                annotations=[],
                audio=None,
                function_call=FunctionCall(
                    arguments='{"quantity":2,"pizza_name":"Hawaii","address":"123 Fakestreet"}',
                    name='place_order'
                ),
                tool_calls=None
            ),
            content_filter_results={}
        )
    ],
    created=1775706494,
    model='gpt-4.1-2025-04-14',
    object='chat.completion',
    service_tier=None,
    system_fingerprint='fp_7a7fd0eb44',
    usage=CompletionUsage(
        completion_tokens=28,
        prompt_tokens=167,
        total_tokens=195,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    ),
    prompt_filter_results=[
        {
            'prompt_index': 0,
            'content_filter_results': {
                'hate': {'filtered': False, 'severity': 'safe'},
                'jailbreak': {'detected': False, 'filtered': False},
                'self_harm': {'filtered': False, 'severity': 'safe'},
                'sexual': {'filtered': False, 'severity': 'safe'},
                'violence': {'filtered': False, 'severity': 'safe'}
            }
        }
    ]
)

# Lets create a fake database 

In [20]:
fake_db = {
    "pizzas": {
        "Hawaii": {"price":15.00, "ingredients": ["ham", "pineapple", "cheese"]},
        "Margherita": {"price":10.00, "ingredients": ["tomato", "mozzarella", "basil"]},
        "Pepperoni": {"price": 12.50, "ingredients": ["pepperoni", "mozzarella", "tomato sauce"]},
        "Veggie": {"price": 11.00, "ingredients": ["bell peppers", "onions", "olives", "mushrooms"]}
    },
    "orders": []
}

In [22]:
import json

def get_pizza_info(pizza_name):
    pizza = fake_db["pizzas"].get(pizza_name)

    if not pizza:
        return f"No information available for pizza: {pizza_name}"
    
    return {"name": pizza_name, "price": pizza["price"], "ingredients": pizza["ingredients"]}

def place_order(pizza_name, quantity, address):
    if pizza_name not in fake_db["pizzas"]:
        return f"We don't have {pizza_name} pizza!"
    
    if quantity < 1:
        return "You must order at least one pizza."
    
    order_id = len(fake_db["orders"]) + 1
    order = {
        "order_id": order_id,
        "pizza_name": pizza_name,
        "quantity": quantity,
        "address": address,
        "total_price": fake_db["pizzas"][pizza_name]["price"] * quantity
    }

    fake_db["orders"].append(order)

    return f"Order placed successfully! Your order ID is {order_id}. Total price is ${order['total_price']}."

response = openai.chat.completions.create(
    model = AZURE_OPENAI_API_DEPLOYMENT_4O,
    messages = [
        {"role": "system", "content": "You are an AI Chatbot having a conversation with a human."},
        {"role": "user", "content": "I want to order two pizza hawaii to 123 Fakestreet"}
    ],
    functions=functions
)

print(response)

print("---------------------------------")
print(response.choices[0].message.function_call.name)
print(response.choices[0].message.function_call.arguments)


function_name = response.choices[0].message.function_call.name
arguments = json.loads(response.choices[0].message.function_call.arguments)
                       
response = place_order(**arguments)

print(response)


[04/09/26 15:00:39] INFO     HTTP Request: POST                                                     ]8;id=578500;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=124674;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

ChatCompletion(
    id='chatcmpl-DSdfTzstQadghMrzEwhj2iqIWhX6y',
    choices=[
        Choice(
            finish_reason='function_call',
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content=None,
                refusal=None,
                role='assistant',
                annotations=[],
                audio=None,
                function_call=FunctionCall(
                    arguments='{"quantity":2,"pizza_name":"Hawaii","address":"123 Fakestreet"}',
                    name='place_order'
                ),
                tool_calls=None
            ),
            content_filter_results={}
        )
    ],
    created=1775718039,
    model='gpt-4.1-2025-04-14',
    object='chat.completion',
    service_tier=None,
    system_fingerprint='fp_7a7fd0eb44',
    usage=CompletionUsage(
        completion_tokens=28,
        prompt_tokens=167,
        total_tokens=195,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    ),
    prompt_filter_results=[
        {
            'prompt_index': 0,
            'content_filter_results': {
                'hate': {'filtered': False, 'severity': 'safe'},
                'jailbreak': {'detected': False, 'filtered': False},
                'self_harm': {'filtered': False, 'severity': 'safe'},
                'sexual': {'filtered': False, 'severity': 'safe'},
                'violence': {'filtered': False, 'severity': 'safe'}
            }
        }
    ]
)

---------------------------------

place_order

{"quantity":2,"pizza_name":"Hawaii","address":"123 Fakestreet"}

Order placed successfully! Your order ID is 2. Total price is $30.0.

In [29]:
class ChatBot:

    def __init__(self, database):
        self.fake_db = database
    
    def chat(self, query):
        initial_response = self.make_openai_request(query)

        message = initial_response.choices[0].message

        if message.function_call:
            function_name = message.function_call.name
            arguments = json.loads(message.function_call.arguments)

            function_response = getattr(self, function_name)(**arguments)

            follow_up_response = self.make_follow_up_request(query, message, function_name, function_response)
            return follow_up_response.choices[0].message.content
        
        else:
            return message.content
        
    def make_openai_request(self, query):
        response = openai.chat.completions.create(
            model=AZURE_OPENAI_API_DEPLOYMENT_4O,
            messages=[{"role":"user", "content": query}],
            functions=functions
        )

        return response
    
    def make_follow_up_request(self, query, initial_message, function_name, function_response):
        response = openai.chat.completions.create(
            model=AZURE_OPENAI_API_DEPLOYMENT_4O,
            messages=[
                {"role": "user", "content": query},
                initial_message,
                {
                    "role": "function",
                    "name": function_name,
                    "content": function_response
                }
            ]
        )
        return response
    
    def place_order(self, pizza_name, quantity, address):
        if pizza_name not in self.fake_db["pizzas"]:
            return f"We don't have {pizza_name} pizza!"
        
        if quantity < 1:
            return "You must order at least one pizza."
        
        order_id = len(self.fake_db["orders"]) + 1
        order = {
            "order_id": order_id,
            "pizza_name": pizza_name,
            "quantity": quantity,
            "address": address,
            "total_price": self.fake_db["pizzas"][pizza_name]["price"] * quantity
        }

        self.fake_db["orders"].append(order)

        return f"Order placed successfully! Your order ID is {order_id}. Total price is ${order['total_price']}."
    
    def get_pizza_info(self, pizza_name):
        if pizza_name in self.fake_db["pizzas"]:
            pizza = self.fake_db["pizzas"][pizza_name]
            return f"Pizza: {pizza['name']}, Price: ${pizza['price']}"
        else:
            return f"We don't have information about {pizza_name} pizza."
        
database = {
    "pizzas": {
        "Hawaii": {
            "name": "Hawaii",
            "price": 15.0
        },
        "Margherita": {
            "name": "Margherita",
            "price": 12.0
        }
    },
    "orders": []
}


    


In [30]:
bot = ChatBot(database=database)

bot.chat("I want to order one pizza Margherita to 124 Fakestreet")
response = bot.chat("I want to order two pizza Hawaii to 123 Fakestreet")

response



[04/09/26 15:46:18] INFO     HTTP Request: POST                                                     ]8;id=470863;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=718644;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/09/26 15:46:19] INFO     HTTP Request: POST                                                     ]8;id=189246;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=304148;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/09/26 15:46:20] INFO     HTTP Request: POST                                                     ]8;id=782645;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=549183;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/09/26 15:46:21] INFO     HTTP Request: POST                                                     ]8;id=224021;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=582614;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

'Your order for two Hawaii pizzas to 123 Fakestreet has been placed successfully! The total price is $30.00. If you need delivery updates or want to add anything else, just let me know.'